# MNIST: Semantics-aware adversarial examples using EBM + projected Langevin MCMC

**Goal:** Start with a correctly classified MNIST **1**, learn low energy for its semantic-preserving transformations, and use stochastic Langevin updates to search for an image classified as **2**. Visualize the result and perturbation.

This is an **educational, simplified experiment**, inspired by [Zhang, Zhang & Wischik, NeurIPS 2024](https://arxiv.org/abs/2306.00353), **not an exact replication**. In particular, the EBM is trained with a stable *contrastive surrogate and short-run negative Langevin chains*, rather than exact EBM maximum likelihood; we use affine/smooth-warp augmentation instead of exact TPS; and our CNN is ordinarily trained, not MadryNet.

The potential and projected Langevin update are

$$V(x)=\lambda_E E_\theta(x)+cL_{\rm target}(x,2)+\lambda_{\rm pix}\|x-x_0\|_2^2,$$
$$x_{k+1}=\Pi_{[0,1]^{784}}\Big(x_k-h\nabla_x V(x_k)+\sqrt{2hT}\,\xi_k\Big),\quad\xi_k\sim\mathcal N(0,I).$$

At $T=1$, this has the usual ULA noise scale; at the smaller demonstration temperature below, the ideal **unconstrained** target is proportional to $e^{-V/T}$. Pixel clipping and finite steps introduce bias. **Attack success does not prove semantic preservation.** Inspect the images.

**Colab:** Runtime → Change runtime type → **T4 GPU** (recommended). Run cells in order. All code uses PyTorch, torchvision, numpy, and matplotlib.

In [ ]:
import math, random, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

SOURCE_DIGIT = 1
TARGET_DIGIT = 2
CLF_EPOCHS = 2
ENERGY_STEPS = 260 if DEVICE.type == 'cuda' else 70
ATTACK_STEPS = 400 if DEVICE.type == 'cuda' else 180
N_CHAINS = 24 if DEVICE.type == 'cuda' else 8

## 1. Download MNIST and train a victim CNN

The attack gradients are taken from this CNN, so this is a **white-box demonstration**. Transfer to unseen classifiers would be a separate black-box evaluation.

In [ ]:
train_ds = datasets.MNIST('./mnist_data', train=True, download=True, transform=transforms.ToTensor())
test_ds  = datasets.MNIST('./mnist_data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2,
                          pin_memory=(DEVICE.type == 'cuda'))
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2,
                         pin_memory=(DEVICE.type == 'cuda'))

class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*7*7, 128),
                                  nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x):
        return self.head(self.conv(x))

victim = DigitCNN().to(DEVICE)
clf_path = Path('mnist_cnn_ebm_demo.pt')
if clf_path.exists():
    victim.load_state_dict(torch.load(clf_path, map_location=DEVICE, weights_only=True))
    print('Loaded saved CNN.')
else:
    opt = torch.optim.Adam(victim.parameters(), lr=1e-3)
    for epoch in range(CLF_EPOCHS):
        victim.train()
        total_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f'CNN epoch {epoch+1}/{CLF_EPOCHS}'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(victim(imgs), labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f'CNN epoch {epoch+1}: avg minibatch loss={total_loss/len(train_loader):.4f}')
    torch.save(victim.state_dict(), clf_path)

victim.eval()
for p in victim.parameters(): p.requires_grad_(False)
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        guesses = victim(imgs.to(DEVICE)).argmax(1).cpu()
        correct += int((guesses == labels).sum())
        total += len(labels)
print(f'Victim test accuracy: {correct/total:.2%}')

## 2. Pick one held-out digit 1 and create semantic-preserving transformations

Positive examples for the EBM are mild translations, rotations, scale changes, and smooth elastic warps of **this one source image**. This is the model's operational notion of "semantic preservation"; it needs human validation.

In [ ]:
candidate_ids = (test_ds.targets == SOURCE_DIGIT).nonzero(as_tuple=True)[0][:350]
candidate_imgs = torch.stack([test_ds[int(i)][0] for i in candidate_ids]).to(DEVICE)
with torch.no_grad():
    logits = victim(candidate_imgs)
    probs = logits.softmax(1)[:, SOURCE_DIGIT]
    is_right = logits.argmax(1) == SOURCE_DIGIT
    probs = probs.masked_fill(~is_right, -1)
    chosen_local = int(probs.argmax().item())
source_id = int(candidate_ids[chosen_local])
x0 = test_ds[source_id][0].unsqueeze(0).to(DEVICE)
print(f'Source test index={source_id}; label={SOURCE_DIGIT}; CNN confidence={probs[chosen_local].item():.3f}')

def augment_source(x, count):
    # Generate modest affine and smoothly warped variants of a (1,1,28,28) source.
    x = x.expand(count, -1, -1, -1)
    device = x.device
    angles = (torch.rand(count, device=device)*2-1) * (13*math.pi/180)
    scales = 0.96 + 0.08*torch.rand(count, device=device)
    theta = torch.zeros(count, 2, 3, device=device)
    theta[:, 0, 0] = scales * angles.cos()
    theta[:, 0, 1] = -scales * angles.sin()
    theta[:, 1, 0] = scales * angles.sin()
    theta[:, 1, 1] = scales * angles.cos()
    theta[:, 0, 2] = (torch.rand(count, device=device)*2-1)*0.12
    theta[:, 1, 2] = (torch.rand(count, device=device)*2-1)*0.12
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    # Smooth, small deformation; illustrative alternative to TPS.
    warp = torch.randn(count, 2, 5, 5, device=device) * 0.016
    warp = F.interpolate(warp, size=(28, 28), mode='bicubic', align_corners=False)
    grid = grid + warp.permute(0, 2, 3, 1)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='zeros').clamp(0, 1)

with torch.no_grad():
    examples = augment_source(x0, 10).cpu()
fig, ax = plt.subplots(2, 6, figsize=(11, 4))
for a in ax.flat: a.axis('off')
ax[0,0].imshow(x0[0,0].cpu(), cmap='gray', vmin=0, vmax=1)
ax[0,0].set_title('Original 1')
for k in range(10):
    a = ax.flat[k+1]
    a.imshow(examples[k,0], cmap='gray', vmin=0, vmax=1)
    a.set_title('Augmented')
plt.tight_layout(); plt.show()

## 3. Train a scalar-output EBM

The neural network returns one real energy $E_\theta(x)$. We want **lower energy** for held-out semantic transformations and **higher energy** for other digits / short-run negative samples.

For a robust Colab teaching demonstration, the optimization uses a *contrastive energy classification objective*: softplus penalties on positive/negative energies, with negatives partly refreshed by short-run Langevin sampling from the current EBM. This is **not exact EBM maximum-likelihood training**, whose ideal gradient is $\mathbb E_{p_{\rm data}}\nabla_\theta E-\mathbb E_{p_\theta}\nabla_\theta E$. Short-run chains and a hand-picked negative mixture do not yield exact model samples. They still let us learn and visualize an energy landscape and then differentiate it during an adversarial Langevin attack.

In [ ]:
class EnergyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.SiLU(), nn.AvgPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.SiLU(), nn.AvgPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7, 96), nn.SiLU(), nn.Linear(96, 1)
        )
    def forward(self, x):
        return self.net(x).flatten()

ebm = EnergyCNN().to(DEVICE)
wrong_ids = (train_ds.targets != SOURCE_DIGIT).nonzero(as_tuple=True)[0]
replay = None

# Short-run projected Langevin negative phase (for training only).
def negative_langevin(initial, steps=10, h=0.005):
    x = initial.detach()
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        e = ebm(x).sum()
        grad_x, = torch.autograd.grad(e, x)
        with torch.no_grad():
            x = (x - h*grad_x + math.sqrt(2*h)*torch.randn_like(x)).clamp_(0,1)
    return x.detach()

def draw_initial_negatives(batch_size):
    global replay
    k = batch_size//2
    idx = wrong_ids[torch.randint(0, len(wrong_ids), (k,))]
    other = train_ds.data[idx].unsqueeze(1).to(DEVICE, dtype=torch.float32)/255.0
    if replay is None:
        rest = torch.rand(batch_size-k, 1, 28, 28, device=DEVICE)
    else:
        chosen = torch.randint(0, len(replay), (batch_size-k,))
        rest = replay[chosen].to(DEVICE)
    return torch.cat([other, rest], dim=0)

opt_e = torch.optim.Adam(ebm.parameters(), lr=1e-3)
energy_history = []
BATCH_E = 48 if DEVICE.type == 'cuda' else 16
for i in tqdm(range(ENERGY_STEPS), desc='Training semantic EBM'):
    positive = augment_source(x0, BATCH_E).detach()
    neg_init = draw_initial_negatives(BATCH_E)
    negative = negative_langevin(neg_init, steps=10, h=0.005)
    # Small replay bank; refresh it with newly sampled negatives.
    new_replay = negative.cpu()
    replay = new_replay if replay is None else torch.cat([replay, new_replay], dim=0)[-384:]

    pos_e, neg_e = ebm(positive), ebm(negative)
    # Stable contrastive energy-ranking surrogate + energy regularization.
    loss = (F.softplus(1 + pos_e).mean()
            + F.softplus(1 - neg_e).mean()
            + 0.002*(pos_e.square().mean() + neg_e.square().mean()))
    opt_e.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(ebm.parameters(), 5.0)
    opt_e.step()
    energy_history.append([float(loss.detach()), float(pos_e.mean().detach()), float(neg_e.mean().detach())])
    if (i+1) % 50 == 0:
        print(f'step {i+1}: loss {loss.item():.3f}, E(positive) {pos_e.mean().item():.2f}, E(negative) {neg_e.mean().item():.2f}')

torch.save(ebm.state_dict(), 'semantic_ebm_demo.pt')
ebm.eval()
for p in ebm.parameters(): p.requires_grad_(False)
plt.plot([r[1] for r in energy_history], label='Training positive energy')
plt.plot([r[2] for r in energy_history], label='Training negative energy')
plt.xlabel('Training step'); plt.ylabel('Mean energy'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 4. Check that the EBM learned something useful

Compare **unseen positive transformations** with a fresh sample of other-digit images. A useful *ranking* is positive energy below negative energy. This alone does **not** establish that the EBM has learned the full semantic data distribution.

In [ ]:
with torch.no_grad():
    pos_val = augment_source(x0, 200)
    idx = wrong_ids[torch.randint(0, len(wrong_ids), (200,))]
    neg_val = train_ds.data[idx].unsqueeze(1).to(DEVICE, dtype=torch.float32) / 255.0
    ep = ebm(pos_val).cpu().numpy()
    en = ebm(neg_val).cpu().numpy()
    original_e = float(ebm(x0).item())
print(f'E(original)={original_e:.3f}  |  mean E(unseen transform)={ep.mean():.3f}'
      f'  |  mean E(other digits)={en.mean():.3f}')
plt.figure(figsize=(8,4))
plt.hist(ep, bins=25, alpha=0.6, label='Held-out semantic transformations')
plt.hist(en, bins=25, alpha=0.6, label='Other MNIST digits')
plt.axvline(original_e, color='black', linestyle='--', label='Original image energy')
plt.xlabel('Energy E_theta(x) (lower is preferred)')
plt.ylabel('Number of images'); plt.legend(); plt.show()

## 5. Targeted projected Langevin adversarial sampling

We minimize the following **potential through stochastic sampling**, not ordinary gradient descent:

$$V(x)=\lambda_E E_\theta(x)+c\,\mathrm{softplus}\big(\max_{j\neq 2} \ell_j(x)-\ell_2(x)+\kappa\big)+\lambda_{\rm pix}\|x-x_0\|_2^2.$$

The logits $\ell_j$ come from the trained CNN. The softplus margin encourages target class **2**; the pixel penalty is an additional practical stabilizer, not the paper's exact objective. When the target succeeds, keep the lowest-distortion candidate across all chains and recorded steps, rather than assuming the last iterate is best.

The update is a **temperature-scaled, pixel-projected ULA**. This is not exact MCMC sampling from the unmodified $e^{-V}$ Gibbs density when $T\neq1$ or due to finite-step clipping. It's a stochastic optimization / exploratory attack prototype.

In [ ]:
# Tune these if the attack is unsuccessful or distorts the digit too much.
H = 0.012                    # Langevin drift step size
TEMPERATURE = 0.012         # T=1 is standard ULA; small T reduces noise for visual quality
TARGET_WEIGHT = 35.0        # stronger -> more likely targeted success, possibly worse semantics
ENERGY_WEIGHT = 1.2         # stronger -> more influence from learned semantic EBM
PIXEL_WEIGHT = 0.9          # stronger -> discourages large pixel changes
TARGET_MARGIN = 0.5

@torch.no_grad()
def check_candidates(batch):
    logits = victim(batch)
    pred = logits.argmax(dim=1)
    target_prob = logits.softmax(1)[:, TARGET_DIGIT]
    l2 = (batch - x0).flatten(1).norm(p=2, dim=1)
    return pred, target_prob, l2

def run_attack(energy_weight=ENERGY_WEIGHT, n_steps=ATTACK_STEPS):
    # Independent chains let us explore several candidate paths.
    x = (x0.expand(N_CHAINS, -1, -1, -1)
         + 0.015*torch.randn(N_CHAINS,1,28,28,device=DEVICE)).clamp(0,1).detach()
    best = None
    best_score = float('inf')
    best_info = None
    snapshots = {0:x[0].detach().cpu().clone()}
    trace = []
    for step in tqdm(range(1, n_steps+1), desc=f'Projected Langevin (lambda_E={energy_weight})'):
        x = x.detach().requires_grad_(True)
        logits = victim(x)
        target = logits[:,TARGET_DIGIT]
        competitors = logits.clone()
        competitors[:,TARGET_DIGIT] = -1e9
        strongest_other = competitors.max(dim=1).values
        target_loss = F.softplus(strongest_other - target + TARGET_MARGIN)
        pixel_penalty = ((x-x0)**2).flatten(1).sum(dim=1)
        potential = (energy_weight*ebm(x) + TARGET_WEIGHT*target_loss
                     + PIXEL_WEIGHT*pixel_penalty)
        g, = torch.autograd.grad(potential.sum(), x)
        with torch.no_grad():
            x = (x - H*g + math.sqrt(2*H*TEMPERATURE)*torch.randn_like(x)).clamp(0,1)

        if step % 5 == 0 or step == 1 or step == n_steps:
            with torch.no_grad():
                pred, target_prob, l2 = check_candidates(x)
                energy_vals = ebm(x)
                successes = (pred == TARGET_DIGIT).nonzero(as_tuple=True)[0]
                if len(successes):
                    # Best-scoring successful image, using L2 and energy as a mild ranking.
                    scores = l2 + 0.08*(energy_vals - original_e).clamp(min=0)
                    local = int(successes[scores[successes].argmin()].item())
                    score = float(scores[local])
                    if score < best_score:
                        best_score = score
                        best = x[local:local+1].detach().cpu().clone()
                        best_info = dict(step=step, chain=local, p_target=float(target_prob[local]),
                                         l2=float(l2[local]), energy=float(energy_vals[local]))
                trace.append([step, float(target_prob.max()), int(len(successes)), float(l2.mean())])
        if step in {1, 40, 100, 200, n_steps}:
            snapshots[step] = x[0].detach().cpu().clone()
    if best is None:
        with torch.no_grad():
            pred, target_prob, l2 = check_candidates(x)
            local = int(target_prob.argmax())
            best = x[local:local+1].detach().cpu().clone()
            best_info = dict(step=n_steps, chain=local, p_target=float(target_prob[local]),
                             l2=float(l2[local]), energy=float(ebm(x[local:local+1]).item()))
    return best, best_info, np.asarray(trace), snapshots

best_img, info, trace, snapshots = run_attack()
print('Best candidate:', info)
with torch.no_grad():
    orig_logits = victim(x0)
    adv_logits = victim(best_img.to(DEVICE))
    print('Original prediction:', int(orig_logits.argmax()),
          '| P(class=1):', round(float(orig_logits.softmax(1)[0,SOURCE_DIGIT]),4))
    print('Candidate prediction:', int(adv_logits.argmax()),
          '| P(class=2):', round(float(adv_logits.softmax(1)[0,TARGET_DIGIT]),4))
    print('Targeted success:', bool(int(adv_logits.argmax()) == TARGET_DIGIT))

## 6. Inspect the actual adversarial image, added pixels, and trajectory

Inspect the images visually. **The classifier predicting 2 does not by itself mean the attack retained the semantics of 1.** If it visibly turns into a 2, the attack failed the semantic-preservation objective even if the classifier was fooled.

In [ ]:
orig = x0.detach().cpu()[0,0].numpy()
adv = best_img[0,0].numpy()
delta = adv - orig
fig, axes = plt.subplots(1,4,figsize=(13,3.3))
axes[0].imshow(orig,cmap='gray',vmin=0,vmax=1); axes[0].set_title('Original: digit 1')
axes[1].imshow(adv,cmap='gray',vmin=0,vmax=1); axes[1].set_title('Langevin candidate')
lim = max(float(np.abs(delta).max()),1e-5)
im=axes[2].imshow(delta,cmap='seismic',vmin=-lim,vmax=lim)
axes[2].set_title('Signed perturbation')
plt.colorbar(im,ax=axes[2],fraction=.046)
axes[3].imshow(np.abs(delta),cmap='magma',vmin=0)
axes[3].set_title('|Perturbation|')
for a in axes: a.axis('off')
plt.tight_layout();plt.show()

print('Perturbation: L1={:.3f}, L2={:.3f}, L_inf={:.3f}, changed pixels(>0.05)={}'.format(
      np.abs(delta).sum(),np.linalg.norm(delta),np.abs(delta).max(),(np.abs(delta)>.05).sum()))
print('EBM: E(original)={:.3f}; E(adversarial)={:.3f}'.format(original_e, info['energy']))

fig, axes=plt.subplots(1,len(snapshots),figsize=(2.4*len(snapshots),3))
for a,(t,im) in zip(axes,sorted(snapshots.items())):
    a.imshow(im[0],cmap='gray',vmin=0,vmax=1)
    a.set_title(f'Chain 0: step {t}'); a.axis('off')
plt.tight_layout();plt.show()

fig, ax = plt.subplots(1,2,figsize=(11,3))
ax[0].plot(trace[:,0],trace[:,1]);ax[0].set(xlabel='Step',ylabel='Best P(target=2) across chains')
ax[1].plot(trace[:,0],trace[:,2]);ax[1].set(xlabel='Step',ylabel='Number of successful chains')
for a in ax: a.grid(alpha=.2)
plt.tight_layout();plt.show()

# Save the output. These files appear in the Colab Files pane.
from torchvision.utils import save_image
save_image(x0.cpu(),'original_digit1.png')
save_image(best_img,'langevin_candidate.png')
np.save('langevin_perturbation.npy',delta)
print('Saved: original_digit1.png, langevin_candidate.png, langevin_perturbation.npy')

## 7. Optional ablation: does adding the semantic EBM help?

Set `ENERGY_WEIGHT=0` (or run the line below) to compare a classifier-only Langevin attack with the EBM-guided variant. A useful EBM should help semantic validity without destroying attack success; that hypothesis needs independent evaluation.

```python
baseline_img, baseline_info, baseline_trace, _ = run_attack(energy_weight=0.0)
print(baseline_info)
fig, axs = plt.subplots(1,3,figsize=(9,3))
for ax, image, title in zip(axs,[x0.cpu(),baseline_img,best_img],
                            ['Original','No EBM','With semantic EBM']):
    ax.imshow(image[0,0],cmap='gray',vmin=0,vmax=1)
    ax.set_title(title);ax.axis('off')
plt.show()
```

### Troubleshooting

- **No targeted success:** rerun `run_attack` with `TARGET_WEIGHT=55`, `ATTACK_STEPS=600`, or a different source image. More success may mean more visible distortion.
- **Image becomes a different digit:** increase `PIXEL_WEIGHT` or `ENERGY_WEIGHT`, decrease `TARGET_WEIGHT`, and inspect held-out EBM energy separation.
- **EBM energies do not separate:** increase `ENERGY_STEPS`, change the negative mixture, or reconsider the semantic augmentations. The current training is only a prototype.
- **Slow on CPU:** use Colab T4 GPU. Reduced CPU defaults may produce weaker results.
- **Project fidelity:** exact paper-style EBM maximum-likelihood training, TPS, a robust MadryNet surrogate, sample refinement, and transfer tests are additional research steps.

**Interpretation:** This experiment demonstrates how the trained energy gradient and target loss appear in the Langevin drift, how Gaussian noise induces exploration, and how projection keeps pixels valid. It is a **white-box surrogate experiment**, not yet a black-box transferable attack, and it does not establish exact Gibbs sampling.